# 🔧 Speech Emotion Recognition — Preprocessing

## 🎯 Objectif
Préparer les données audio du dataset RAVDESS pour 3 architectures de modèles :
1. **CNN + BiLSTM** : Features acoustiques (MFCC, ZCR, RMS, Chroma, Spectral)
2. **Wav2Vec2** : Waveform brute à 16kHz
3. **HuBERT** : Waveform brute à 16kHz

## 💾 Sorties
- `preprocessed_cnn_bilstm.npz` : Features extraites pour CNN+BiLSTM
- `preprocessed_transformers.npz` : Waveforms pour Wav2Vec2 / HuBERT
- `label_encoder.npy` : Classes émotions encodées

## ⚙️ Configuration RTX 3050
- Durée audio : 3 secondes (padding/truncating)
- Waveform length : 48000 samples @ 16kHz

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 1. Chargement du Dataset

In [ ]:
DATA_PATH = 'RAVDESS'

EMOTION_MAP = {
    '01': 'neutral', '02': 'calm', '03': 'happy', '04': 'sad',
    '05': 'angry', '06': 'fearful', '07': 'disgust', '08': 'surprise'
}

data = []
for actor_folder in sorted(os.listdir(DATA_PATH)):
    actor_path = os.path.join(DATA_PATH, actor_folder)
    if not os.path.isdir(actor_path) or actor_folder == 'audio_speech_actors_01-24':
        continue
    for file in sorted(os.listdir(actor_path)):
        if not file.endswith('.wav'):
            continue
        parts = file.replace('.wav', '').split('-')
        emotion = EMOTION_MAP.get(parts[2])
        actor_id = int(parts[6])
        if emotion:
            data.append({'path': os.path.join(actor_path, file), 'emotion': emotion, 'actor_id': actor_id})

df = pd.DataFrame(data)
print(f'Total fichiers : {len(df)}')
print(df['emotion'].value_counts())

## 2. Label Encoding

In [ ]:
encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['emotion'])

# Sauvegarder les classes
np.save('label_classes.npy', encoder.classes_)
print('Classes :', encoder.classes_)
print('Mapping :', dict(zip(encoder.classes_, range(len(encoder.classes_)))))

## 3. Feature Extraction pour CNN + BiLSTM

Features extraites par fichier :
- **40 MFCCs** : Coefficients cepstraux en fréquence Mel
- **12 Chroma** : Représentation de la tonalité
- **1 ZCR** : Taux de passage par zéro
- **1 RMS** : Énergie du signal
- **1 Spectral Centroid** : Centre de gravité spectral
- **1 Spectral Bandwidth** : Largeur spectrale
- **1 Spectral Rolloff** : Fréquence de rolloff

Total : **57 features** x **130 frames temporelles**

In [ ]:
SR_CNN = 22050
DURATION = 3  # secondes
N_SAMPLES = SR_CNN * DURATION
N_MFCC = 40
MAX_FRAMES = 130  # nombre de frames temporelles fixé

def extract_features_cnn(file_path):
    """Extraire les features acoustiques pour le modèle CNN+BiLSTM."""
    y, sr = librosa.load(file_path, sr=SR_CNN, duration=DURATION)
    
    # Padding si nécessaire
    if len(y) < N_SAMPLES:
        y = np.pad(y, (0, N_SAMPLES - len(y)))
    else:
        y = y[:N_SAMPLES]
    
    # Extraction des features
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)           # (40, T)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr, n_chroma=12)     # (12, T)
    zcr = librosa.feature.zero_crossing_rate(y=y)                      # (1, T)
    rms = librosa.feature.rms(y=y)                                     # (1, T)
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)           # (1, T)
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)         # (1, T)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)             # (1, T)
    
    # Concaténation : (57, T)
    features = np.vstack([mfcc, chroma, zcr, rms, centroid, bandwidth, rolloff])
    
    # Ajuster la dimension temporelle
    if features.shape[1] < MAX_FRAMES:
        features = np.pad(features, ((0, 0), (0, MAX_FRAMES - features.shape[1])))
    else:
        features = features[:, :MAX_FRAMES]
    
    return features  # (57, 130)

print(f'Shape attendue par fichier : (57, {MAX_FRAMES})')
# Test
test_feat = extract_features_cnn(df.iloc[0]['path'])
print(f'Shape r\u00e9elle : {test_feat.shape}')

In [ ]:
# Extraire toutes les features CNN+BiLSTM
X_cnn = []
y_cnn = []

print('Extraction des features pour CNN+BiLSTM...')
for idx, row in tqdm(df.iterrows(), total=len(df)):
    feat = extract_features_cnn(row['path'])
    X_cnn.append(feat)
    y_cnn.append(row['label'])

X_cnn = np.array(X_cnn)      # (N, 57, 130)
y_cnn = np.array(y_cnn)      # (N,)

# Transposer pour le modèle : (N, 130, 57) = (batch, timesteps, features)
X_cnn = np.transpose(X_cnn, (0, 2, 1))

print(f'\nX_cnn shape: {X_cnn.shape}')
print(f'y_cnn shape: {y_cnn.shape}')

In [ ]:
# Normalisation avec StandardScaler (par feature)
n_samples, n_timesteps, n_features = X_cnn.shape

# Reshape pour le scaler
X_flat = X_cnn.reshape(-1, n_features)
scaler = StandardScaler()
X_flat = scaler.fit_transform(X_flat)
X_cnn_normalized = X_flat.reshape(n_samples, n_timesteps, n_features)

print(f'X_cnn_normalized shape: {X_cnn_normalized.shape}')
print(f'Mean (devrait ~0): {X_cnn_normalized.mean():.4f}')
print(f'Std (devrait ~1): {X_cnn_normalized.std():.4f}')

## 4. Préparation des Waveforms pour Wav2Vec2 / HuBERT

Les modèles Wav2Vec2 et HuBERT attendent :
- Audio **mono**, **16kHz**
- Waveform brute (pas de features manuelles)
- Durée fixe : **3 secondes = 48,000 samples**

In [ ]:
SR_TRANSFORMER = 16000
MAX_LENGTH = SR_TRANSFORMER * DURATION  # 48000 samples

def load_waveform(file_path):
    """Charger et pr\u00e9parer un waveform pour les mod\u00e8les transformer."""
    y, sr = librosa.load(file_path, sr=SR_TRANSFORMER, duration=DURATION)
    
    # Padding / Truncating
    if len(y) < MAX_LENGTH:
        y = np.pad(y, (0, MAX_LENGTH - len(y)))
    else:
        y = y[:MAX_LENGTH]
    
    return y  # (48000,)

print(f'Waveform length: {MAX_LENGTH} samples ({DURATION}s @ {SR_TRANSFORMER}Hz)')
test_wav = load_waveform(df.iloc[0]['path'])
print(f'Shape: {test_wav.shape}')

In [ ]:
# Extraire tous les waveforms
X_wav = []
y_wav = []

print('Chargement des waveforms pour Wav2Vec2/HuBERT...')
for idx, row in tqdm(df.iterrows(), total=len(df)):
    waveform = load_waveform(row['path'])
    X_wav.append(waveform)
    y_wav.append(row['label'])

X_wav = np.array(X_wav, dtype=np.float32)  # (N, 48000)
y_wav = np.array(y_wav)                     # (N,)

print(f'\nX_wav shape: {X_wav.shape}')
print(f'y_wav shape: {y_wav.shape}')
print(f'M\u00e9moire: {X_wav.nbytes / 1024**2:.1f} MB')

## 5. Data Augmentation

Techniques d'augmentation audio pour améliorer la généralisation :
- **Noise injection** : Ajout de bruit gaussien
- **Time stretching** : Modification de la vitesse
- **Pitch shifting** : Décalage de tonalité

In [ ]:
def add_noise(y, noise_factor=0.005):
    """Ajouter du bruit gaussien."""
    noise = np.random.randn(len(y))
    return y + noise_factor * noise

def time_stretch(y, rate=1.1):
    """Modifier la vitesse."""
    stretched = librosa.effects.time_stretch(y=y, rate=rate)
    if len(stretched) < len(y):
        stretched = np.pad(stretched, (0, len(y) - len(stretched)))
    return stretched[:len(y)]

def pitch_shift(y, sr, n_steps=2):
    """D\u00e9caler la tonalit\u00e9."""
    return librosa.effects.pitch_shift(y=y, sr=sr, n_steps=n_steps)

# Augmentation uniquement sur le train set (après le split)
print('Fonctions d\'augmentation d\u00e9finies.')
print('L\'augmentation sera appliqu\u00e9e dans le notebook de training.')

## 6. Split Train / Validation / Test

Stratégie de split :
- **Train** : 70%
- **Validation** : 15%
- **Test** : 15%
- Split **stratifié** pour maintenir la distribution des émotions
- **Random state fixe** pour reproductibilité

In [ ]:
RANDOM_STATE = 42

# --- Split pour CNN+BiLSTM ---
X_train_cnn, X_temp_cnn, y_train_cnn, y_temp_cnn = train_test_split(
    X_cnn_normalized, y_cnn, test_size=0.3, random_state=RANDOM_STATE, stratify=y_cnn
)
X_val_cnn, X_test_cnn, y_val_cnn, y_test_cnn = train_test_split(
    X_temp_cnn, y_temp_cnn, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp_cnn
)

print('=== CNN+BiLSTM ===')
print(f'Train: {X_train_cnn.shape[0]} | Val: {X_val_cnn.shape[0]} | Test: {X_test_cnn.shape[0]}')

# --- Split pour Transformers (mêmes indices) ---
X_train_wav, X_temp_wav, y_train_wav, y_temp_wav = train_test_split(
    X_wav, y_wav, test_size=0.3, random_state=RANDOM_STATE, stratify=y_wav
)
X_val_wav, X_test_wav, y_val_wav, y_test_wav = train_test_split(
    X_temp_wav, y_temp_wav, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp_wav
)

print('\n=== Wav2Vec2 / HuBERT ===')
print(f'Train: {X_train_wav.shape[0]} | Val: {X_val_wav.shape[0]} | Test: {X_test_wav.shape[0]}')

In [ ]:
# Vérification de la stratification
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
classes = np.load('label_classes.npy', allow_pickle=True)

for ax, (name, labels) in zip(axes, [('Train', y_train_cnn), ('Val', y_val_cnn), ('Test', y_test_cnn)]):
    unique, counts = np.unique(labels, return_counts=True)
    ax.bar(classes[unique], counts, color=sns.color_palette('Set2', len(unique)))
    ax.set_title(f'{name} ({len(labels)} samples)')
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylabel('Count')

plt.suptitle('V\u00e9rification de la stratification', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Sauvegarde des Données Préprocessées

In [ ]:
# Créer le dossier de sortie
os.makedirs('preprocessed', exist_ok=True)

# --- Sauvegarder CNN+BiLSTM ---
np.savez_compressed('preprocessed/cnn_bilstm_data.npz',
    X_train=X_train_cnn, X_val=X_val_cnn, X_test=X_test_cnn,
    y_train=y_train_cnn, y_val=y_val_cnn, y_test=y_test_cnn
)
print('CNN+BiLSTM data saved.')

# --- Sauvegarder Transformers ---
np.savez_compressed('preprocessed/transformer_data.npz',
    X_train=X_train_wav, X_val=X_val_wav, X_test=X_test_wav,
    y_train=y_train_wav, y_val=y_val_wav, y_test=y_test_wav
)
print('Transformer data saved.')

# --- Taille des fichiers ---
for f in ['preprocessed/cnn_bilstm_data.npz', 'preprocessed/transformer_data.npz']:
    size_mb = os.path.getsize(f) / 1024**2
    print(f'  {f}: {size_mb:.1f} MB')

In [ ]:
# Résumé final
print('=' * 60)
print('PREPROCESSING TERMIN\u00c9')
print('=' * 60)
print(f'\nCNN+BiLSTM :')
print(f'  Input shape  : ({X_train_cnn.shape[1]}, {X_train_cnn.shape[2]}) = (timesteps, features)')
print(f'  Num classes  : {len(np.unique(y_cnn))}')
print(f'  Features     : MFCC(40) + Chroma(12) + ZCR + RMS + Centroid + BW + Rolloff = 57')
print(f'\nWav2Vec2 / HuBERT :')
print(f'  Input shape  : ({X_train_wav.shape[1]},) = 3s @ 16kHz')
print(f'  Num classes  : {len(np.unique(y_wav))}')
print(f'\nSplit sizes :')
print(f'  Train : {X_train_cnn.shape[0]}')
print(f'  Val   : {X_val_cnn.shape[0]}')
print(f'  Test  : {X_test_cnn.shape[0]}')
print(f'\nFichiers sauv\u00e9gard\u00e9s dans preprocessed/')